### Balanced Label Creation
> - cens_dfs == 1 -> event occured (recurrence, death, etc.)
> - cens_dfs == 0 -> censored (no event observed, maybe lost to follow-up or study ended)

We want to focus only on patients with an event occuring (cens_dfs == 1) and compare their DFS time.

In [8]:
import os
import re
import random
from collections import defaultdict

import numpy as np
import pandas as pd
from ete3 import Tree

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report

In [53]:
# From TracerX dataset, we want to create a balanced dataset based on DFS time.
tracerx_df = pd.read_csv("data/tracerX.csv")
tracerx_df.columns = tracerx_df.columns.str.strip()
print(tracerx_df.columns)

threshold_dfs_time = 365*3.5 # Set a threshold for DFS time (e.g., 3.5 year)

# Drop rows with missing dfs_time
tracerx_df = tracerx_df.dropna(subset=['dfs_time'])

# Filter datasets
events_df = tracerx_df[tracerx_df['cens_dfs'] == 1]
non_events_df = tracerx_df[(tracerx_df['dfs_time'] > threshold_dfs_time) & (tracerx_df['cens_dfs'] == 0)]

print(f"Number of events: {len(events_df)}, Number of non-events: {len(non_events_df)}")

# Combine and label
combined_events_df = pd.concat([events_df, non_events_df], ignore_index=True)
combined_events_df['shorter_dfs_balanced'] = (combined_events_df['dfs_time'] < threshold_dfs_time).astype(int)

print(combined_events_df['shorter_dfs_balanced'].value_counts())
print(combined_events_df['shorter_dfs_balanced'].value_counts(normalize=True))

Index(['cruk_id', 'tumour_id_muttable_cruk', 'tumour_id_per_patient', 'age',
       'sex', 'ethnicity', 'cigs_perday', 'years_smoking', 'packyears',
       'smoking_status_merged', 'is.family.lung', 'ECOG_PS', 'pathologyTNM',
       'pT_stage_per_patient', 'pN_stage_per_patient', 'LVI_per_patient',
       'PL_per_patient', 'margin_status_per_patient',
       'size_pathology_per_patient', 'Surgery_type', 'histology_lesion1',
       'histology_lesion1_merged', 'lesion1_sampled', 'histology_lesion2',
       'lesion2_sampled', 'histology_multi_full',
       'histology_multi_full_genomically.confirmed', 'LUAD_pred_subtype',
       'adjuvant_treatment_YN', 'adjuvant_treatment_given',
       'num_cycle_na.added', 'CHMPlatDgName_cleaned', 'CHMOthDgName_cleaned',
       'AdjRadStartTime_manual', 'AdjRadEndTime_manual', 'Recurrence_time_use',
       'newPrim_time_use', 'first_dfs_any_event_rec.or.new.primary',
       'first_event_during_followup', 'cens_os', 'os_time', 'cens_dfs',
       'dfs_ti

### Example Tree and clinical string encodings
>- Newick Tree: ((1,11)8,(14,12)9)root; etc. 
>- Clinical data: ("age_65 sex_M ECOG_PS_1 smoking_20py"...) etc.

For simplicity, we consider only the primary T1 tumor for every patient.

In [69]:
newick_dir = "trees.txt"  #  file containing patient newick trees
clinical_csv = "data/tracerX.csv"
cruk_id_col = "cruk_id"
newick_file_template = "{cruk_id}.newick"  # e.g., CRUK0361.newick
random_seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 60
lr = 1e-3
embed_dim = 32
mem_dim = 64
# optional: explicit clinical features list (uncomment & set) - otherwise numeric columns auto-detected
clinical_feature_cols = None  # e.g. ['age', 'packyears']

In [71]:
import pandas as pd

# Load mapping
tree_id_map = pd.read_csv("tree_tumourid.csv", index_col=0)
tree_id_map.columns = tree_id_map.columns.str.strip()

# Filter out Tumour2
tree_id_map_filtered = tree_id_map[~tree_id_map['x'].str.contains('Tumour2')].reset_index(drop=True)

# Load Newick trees
with open("trees.txt") as f:
    newick_lines = [line.strip() for line in f if line.strip()]

# Build mapping - filtered IDs only
trees_by_crukid = dict(zip(tree_id_map_filtered['x'], newick_lines))

print(f"Loaded {len(trees_by_crukid)} trees after filtering")

Loaded 392 trees after filtering


In [72]:
removed_ids = sorted(set(tree_id_map['x']) - set(tree_id_map_filtered['x']))
print(f"{len(removed_ids)} IDs removed: {removed_ids}")

9 IDs removed: ['CRUK0030_Tumour2', 'CRUK0223_Tumour2', 'CRUK0372_Tumour2', 'CRUK0555_Tumour2', 'CRUK0586_Tumour2', 'CRUK0620_Tumour2', 'CRUK0704_Tumour2', 'CRUK0721_Tumour2', 'CRUK0881_Tumour2']


### Tree-LSTMs
Parsing a Newick tree string into a Tree-LSTM object.

In [84]:
# pip install ete3 torch
from ete3 import Tree
import torch
import torch.nn as nn
import torch.nn.functional as F

In [92]:
class TreeNode:
    def __init__(self, name=None):
        self.name = name
        self.children = []
        self.idx = None

def parse_newick_to_treenode(newick_str):
    ete_tree = Tree(newick_str, format=1)
    
    def build_node(ete_node):
        node = TreeNode(name=ete_node.name if ete_node.name else None)
        node.children = [build_node(child) for child in ete_node.children]
        return node
    
    return build_node(ete_tree)

In [93]:
# Helper function to convert TreeNode to a list of indices
def collect_names(node, names):
    if node.name:
        names.add(node.name)
    for c in node.children:
        collect_names(c, names)

def assign_indices(node, vocab):
    node.idx = vocab.get(node.name, 0)
    for c in node.children:
        assign_indices(c, vocab)

In [94]:
# TreeLSTM dataset class
class ChildSumTreeLSTMCell(nn.Module):
    def __init__(self, in_dim, mem_dim):
        super().__init__()
        self.ioux = nn.Linear(in_dim, 3 * mem_dim)
        self.iouh = nn.Linear(mem_dim, 3 * mem_dim)
        self.fx = nn.Linear(in_dim, mem_dim)
        self.fh = nn.Linear(mem_dim, mem_dim)

    def forward(self, inputs, child_c, child_h):
        if child_h:
            h_sum = torch.sum(torch.stack(child_h, dim=0), dim=0)
        else:
            h_sum = torch.zeros(inputs.size(0), self.iouh.in_features, device=inputs.device)

        iou = self.ioux(inputs) + self.iouh(h_sum)
        i, o, u = torch.chunk(torch.sigmoid(iou), 3, dim=1)
        u = torch.tanh(u)

        f = []
        for h, c in zip(child_h, child_c):
            f_child = torch.sigmoid(self.fx(inputs) + self.fh(h))
            f.append(f_child * c)

        c = i * u + sum(f) if f else i * u
        h = o * torch.tanh(c)
        return c, h

class TreeLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, mem_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.cell = ChildSumTreeLSTMCell(embed_dim, mem_dim)

    def forward(self, tree):
        idx = torch.tensor([tree.idx if tree.name else 0], dtype=torch.long, device=self.emb.weight.device)
        inputs = self.emb(idx)
        child_c, child_h = [], []
        for child in tree.children:
            c, h = self.forward(child)
            child_c.append(c)
            child_h.append(h)
        c, h = self.cell(inputs, child_c, child_h)
        return c, h

In [87]:
class TreeLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, mem_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.cell = ChildSumTreeLSTMCell(embed_dim, mem_dim)

    def forward(self, tree):
        # Get embeddings for this node
        if tree.name is None:
            # Placeholder for internal nodes with no explicit label
            idx = torch.tensor([0], dtype=torch.long)
        else:
            idx = torch.tensor([tree.idx], dtype=torch.long)

        inputs = self.emb(idx)

        # Recursively compute child states
        child_c, child_h = [], []
        for child in tree.children:
            c, h = self.forward(child)
            child_c.append(c)
            child_h.append(h)

        c, h = self.cell(inputs, child_c, child_h)
        return c, h

In [83]:
# Display first few trees
for cruk_id, newick in list(trees_by_crukid.items())[:5]:
    print(f"{cruk_id}: {newick}")
    

CRUK0005: (((((((8:24)18:7)12:3,10:10)13:13)9:159,((((17:4)21:89)5:87)19:2)20:178)3:7)4:82,1:203)2;
CRUK0057: ((4:23)2:999,((6:1)5:24)3:1007)1;
CRUK0039: ((4:34)2:958,(5:24)3:971)1;
CRUK0196: (((7:2,8:2)6:12)2:631,(5:4)4:636)1;
CRUK0023: ((((12:11)4:13)13:20)3:195,((8:1e-06)9:2,(10:1e-06)11:3)7:209,2:189)1;


In [95]:
# Building vocabulary from tree names
names = set()
for newick_str in trees_by_crukid.values():
    root = parse_newick_to_treenode(newick_str)
    collect_names(root, names)
vocab = {name: i+1 for i, name in enumerate(sorted(names))}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TreeLSTM(vocab_size=len(vocab)+1, embed_dim=16, mem_dim=32).to(device)
model.eval()

# Generate embeddings for all trees
patient_embeddings = {}
with torch.no_grad():
    for cruk_id, newick_str in trees_by_crukid.items():
        root = parse_newick_to_treenode(newick_str)
        assign_indices(root, vocab)
        _, h = model(root)
        patient_embeddings[cruk_id] = h.squeeze(0).cpu()

# Convert embeddings to DataFrame
df_embeddings = pd.DataFrame.from_dict(
    {pid: emb.numpy() for pid, emb in patient_embeddings.items()},
    orient="index"
).reset_index().rename(columns={"index": "cruk_id"})

In [96]:
# Merge with clinical data
tracerx_df = pd.read_csv("data/tracerX.csv")
tracerx_df.columns = tracerx_df.columns.str.strip()

threshold_dfs_time = 365 * 3.5
events_df = tracerx_df[tracerx_df['cens_dfs'] == 1]
non_events_df = tracerx_df[(tracerx_df['dfs_time'] > threshold_dfs_time) & (tracerx_df['cens_dfs'] == 0)]
combined_df = pd.concat([events_df, non_events_df], ignore_index=True)

combined_df['shorter_dfs_balanced'] = 0
combined_df.loc[(combined_df['dfs_time'] < threshold_dfs_time), 'shorter_dfs_balanced'] = 1

# Merge embeddings with labels
final_df = df_embeddings.merge(
    combined_df[['cruk_id', 'shorter_dfs_balanced']],
    on="cruk_id",
    how="inner"
)

print("Final dataset shape:", final_df.shape)
print(final_df.head())

# Save for downstream modeling
final_df.to_csv("patient_embeddings_with_labels.csv", index=False)

Final dataset shape: (339, 34)
    cruk_id         0         1         2         3         4         5  \
0  CRUK0005  0.260493  0.248584  0.122941  0.259172  0.303271  0.283671   
1  CRUK0057  0.410762  0.197672  0.113944  0.287756  0.344419  0.191199   
2  CRUK0039  0.396152  0.196684  0.108853  0.280514  0.338964  0.184203   
3  CRUK0196  0.467913  0.207125  0.113710  0.306655  0.396264  0.202193   
4  CRUK0023  0.560451  0.241935  0.142786  0.357448  0.404039  0.231594   

          6         7         8  ...        23        24        25        26  \
0  0.167181  0.187055  0.220782  ...  0.420797  0.258536  0.377101  0.255191   
1  0.194983  0.253612  0.166223  ...  0.220460  0.224408  0.211794  0.241770   
2  0.189908  0.250341  0.157567  ...  0.212551  0.224746  0.198700  0.238518   
3  0.197440  0.207123  0.155221  ...  0.276804  0.245076  0.227995  0.239082   
4  0.259086  0.315402  0.184442  ...  0.317190  0.291248  0.285754  0.303269   

         27        28        29      